# Inspect optimization

```
rsync -avm --include='*/' --include='*.sql*' --exclude='*' jhummel@login.delftblue.tudelft.nl:../../scratch/jhummel/tip_clearance/data/co-design/ ./data/co-design/ --dry-run
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from weis.visualization.utils import load_OMsql

plt.style.use("journal.mplstyle")

%matplotlib widget

In [ ]:
logs_to_load = {
    "Baseline": "../../../data/co-design/baseline/baseline.sql",
    "Free yaw": "../../../data/co-design/free_yaw/free_yaw.sql",
    "Zero yaw": "../../../data/co-design/zero_yaw/zero_yaw.sql",
}

# Load all datasets.
all_data_dicts = {}
for log_name, log_fmt in logs_to_load.items():
    all_data_dicts[log_name] = load_OMsql(log_fmt)
    print(f"Loaded {log_name}: {all_data_dicts[log_name].keys()}")

In [ ]:
# Let's define how we load, scale, and label the data, then make a dataframe.
all_outputs = {
    # ROSCO variables.
    "TCIPC_MaxTipDeflection": {
        "key": "tune_rosco_ivc.TCIPC_MaxTipDeflection",
        "scaling": lambda x: x[0],
        "label": "TCIPC reference (m)",
    },
    "TCIPC_TowerClearanceReference": {
        "key": "tune_rosco_ivc.TCIPC_MaxTipDeflection",
        "scaling": lambda x: 30 - x[0],
        "label": "TCIPC reference (m)",
    },
    "ps_percent": {
        "key": "tune_rosco_ivc.ps_percent",
        "scaling": lambda x: x[0],
        "label": "Peak shaving (-)",
    },
    "max_thrust_percent": {
        "key": "tune_rosco_ivc.ps_percent",
        "scaling": lambda x: 100 * x[0],
        "label": "Max thrust (%)",
    },
    "TCIPC_nHarmonics": {
        "key": "tune_rosco_ivc.TCIPC_nHarmonics",
        "scaling": lambda x: x[0],
        "label": "Number of harmonics",
    },
    "TCIPC_ZeroYawDeflection": {
        "key": "tune_rosco_ivc.TCIPC_ZeroYawDeflection",
        "scaling": lambda x: x[0],
        "label": "Zero yaw deflection",
    },
    "TCIPC_MaxPitchAmplitude": {
        "key": "tune_rosco_ivc.TCIPC_MaxPitchAmplitude",
        "scaling": lambda x: np.round(np.rad2deg(x[0]), 1),
        "label": "TCIPC max pitch amplitude (deg)",
    },
    # Design variables
    "cone": {
        "key": "hub.cone",
        "scaling": lambda x: np.rad2deg(x[0]),
        "label": "Cone angle (deg)",
    },
    "tilt": {
        "key": "nacelle.uptilt",
        "scaling": lambda x: np.rad2deg(x[0]),
        "label": "Tilt angle (deg)",
    },
    "overhang": {
        "key": "nacelle.overhang",
        "scaling": lambda x: x[0],
        "label": "Overhang (m)",
    },
    # Objective.
    "aep": {
        "key": "aeroelastic.AEP",
        "scaling": lambda x: 1e-6 * x[0],
        "label": "AEP (GWh)",
    },
    # Constraints.
    "max_eff_TipDxc_towerPassing": {
        "key": "aeroelastic.max_eff_TipDxc_towerPassing",
        "scaling": lambda x: x[0],
        "label": "Max effective TipDxc tower passing (m)",
    },
    # "max_TipDxc_towerPassing": {
    #     "key": "aeroelastic.max_TipDxc_towerPassing",
    #     "scaling": lambda x: x[0],
    #     "label": "Max TipDxc tower passing (m)",
    # },
    "max_TipDxc_towerPassing_DLC": {
        "key": "aeroelastic.max_TipDxc_towerPassing_DLC",
        "scaling": lambda x: str(x[0]),
        "label": "Max TipDxc tower passing DLC (-)",
    },
    "max_TipDxc_towerPassing_U": {
        "key": "aeroelastic.max_TipDxc_towerPassing_U",
        "scaling": lambda x: x[0],
        "label": "Max TipDxc tower passing U (m/s)",
    },
    # "tower_clearance": {
    #     "key": "aeroelastic.max_TipDxc_towerPassing",
    #     "scaling": lambda x: 30 - x[0],
    #     "label": "Tower clearance (m)",
    # },
    "eff_tower_clearance": {
        "key": "aeroelastic.max_eff_TipDxc_towerPassing",
        "scaling": lambda x: 30 - x[0],
        "label": "Tower clearance (m)",
    },
    # Structural loads.
    "max_eff_TwrBsMyt": {
        "key": "aeroelastic.max_eff_TwrBsMyt",
        "scaling": lambda x: x[0] / 1000,
        "label": "Tower ultimate (MNm)",
    },
    # "max_TwrBsMyt": {
    #     "key": "aeroelastic.max_TwrBsMyt",
    #     "scaling": lambda x: x[0] / 1000,
    #     "label": "Max tower base Myt (MNm)",
    # },
    "max_TwrBsMyt_DLC": {
        "key": "aeroelastic.max_TwrBsMyt_DLC",
        "scaling": lambda x: str(x[0]),
        "label": "Max tower base Myt DLC (-)",
    },
    "max_TwrBsMyt_U": {
        "key": "aeroelastic.max_TwrBsMyt_U",
        "scaling": lambda x: x[0],
        "label": "Max tower base Myt U (m/s)",
    },
    "DEL_TwrBsMyt": {
        "key": "aeroelastic.DEL_TwrBsMyt",
        "scaling": lambda x: x[0] / 1000,
        "label": "Tower fatigue (MNm)",
    },
    "damage_tower_base": {
        "key": "aeroelastic.damage_tower_base",
        "scaling": lambda x: x[0],
        "label": "Tower base damage (-)",
    },
    "max_eff_RootMyb": {
        "key": "aeroelastic.max_eff_RootMyb",
        "scaling": lambda x: x[0] / 1000,
        "label": "Blade ultimate (MNm)",
    },
    # "max_RootMyb": {
    #     "key": "aeroelastic.max_RootMyb",
    #     "scaling": lambda x: x[0] / 1000,
    #     "label": "Max root Myb (MNm)",
    # },
    "max_RootMyb_DLC": {
        "key": "aeroelastic.max_RootMyb_DLC",
        "scaling": lambda x: str(x[0]),
        "label": "Max root Myb DLC (-)",
    },
    "max_RootMyb_U": {
        "key": "aeroelastic.max_RootMyb_U",
        "scaling": lambda x: x[0],
        "label": "Max root Myb U (m/s)",
    },
    "DEL_RootMyb": {
        "key": "aeroelastic.DEL_RootMyb",
        "scaling": lambda x: x[0] / 1000,
        "label": "Blade fatigue (MNm)",
    },
    "hub_Mxyz": {
        "key": "aeroelastic.hub_Mxyz",
        "scaling": lambda x: np.linalg.norm(x) / 1000,
        "label": "Hub moment magnitude (MNm)",
    },
    # Control activity.
    "max_pitch_rate_sim": {
        "key": "aeroelastic.max_pitch_rate_sim",
        "scaling": lambda x: x[0],
        "label": "Max pitch rate (deg/s)",
    },
    "max_pitch_rate_sim_DLC": {
        "key": "aeroelastic.max_pitch_rate_sim_DLC",
        "scaling": lambda x: str(x[0]),
        "label": "Max pitch rate DLC (-)",
    },
    "max_pitch_rate_sim_U": {
        "key": "aeroelastic.max_pitch_rate_sim_U",
        "scaling": lambda x: x[0],
        "label": "Max pitch rate U (m/s)",
    },
    "avg_pitch_travel": {
        "key": "aeroelastic.avg_pitch_travel",
        "scaling": lambda x: x[0],
        "label": "Avg pitch travel (deg)",
    },
}

# Build dataframe from mapping for each log.
labels = {short: info["label"] for short, info in all_outputs.items()}
all_dfs = []

for log_name, data_dict in all_data_dicts.items():
    df_dict = {}
    for short_label, info in all_outputs.items():
        data = data_dict[info["key"]]
        scaled_data = list(map(info["scaling"], data))
        df_dict[short_label] = scaled_data

    df_temp = pd.DataFrame(df_dict)
    df_temp["log_name"] = log_name
    all_dfs.append(df_temp)

# Combine all dataframes.
df = pd.concat(all_dfs, ignore_index=True)
print(f"Combined dataframe shape: {df.shape}")
df

In [ ]:
# Overview plot of the design variables, objective, and constraints.
objectives = ["aep"]
design_variables = [
    "max_thrust_percent",
    "TCIPC_MaxTipDeflection",
    "cone",
    "tilt",
    "overhang",
]
constraints = [
    "eff_tower_clearance",
    "max_eff_RootMyb",
    "max_eff_TwrBsMyt",
    "DEL_RootMyb",
    "DEL_TwrBsMyt",
]

outputs = objectives + design_variables + constraints
log_names = df["log_name"].unique()

In [ ]:
n_outputs = len(outputs)
fig, axs = plt.subplots(n_outputs, 1, figsize=(5, 2 * n_outputs))
for log_name in log_names:
    df_log = df[df["log_name"] == log_name]

    for i, output in enumerate(outputs):
        out = df_log[output]
        axs[i].plot(range(len(out)), out.values, label=log_name)
        axs[i].set_ylabel(labels[output])

        axs[i].legend()

        # axs[i].set_ylim((0, df[output].max()*1.05))
        # axs[i].set_ylim(out.quantile(0.02), out.quantile(0.98))

axs[0].set_ylim(81, 84)